# 🦥 BƯỚC 2: UNSLOTH CODING FINE-TUNING PIPELINE (Chạy trên Colab L4 / A100 / T4)
Notebook này sử dụng **Unsloth** để huấn luyện / tinh chỉnh mô hình lập trình với dữ liệu code tùy chỉnh (Custom Coding Dataset), tăng tốc độ train 2x-5x và tiết kiệm 80% VRAM nhờ QLoRA.

### 📌 Quy trình:
1. Cài đặt Unsloth và thư viện tương thích.
2. Nạp mô hình đã uncensor từ Bước 1 (trên Google Drive hoặc nạp trực tiếp từ Hugging Face).
3. Nạp tập dữ liệu huấn luyện coding (Instruction/Input/Output).
4. Huấn luyện QLoRA 4-bit siêu tốc với triton kernels.
5. Kiểm tra thử suy luận (Inference Test) để kiểm tra chất lượng.
6. Merge LoRA adapter vào mô hình gốc và lưu lại Google Drive (`/content/drive/MyDrive/ai_coding_models_finetuned/`).

In [ ]:
# @title 1. Cài đặt Unsloth & Dependencies
import torch
!nvidia-smi

# Cài đặt Unsloth nhanh chóng
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes datasets

In [ ]:
# @title 2. Kết nối Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
UNCENSORED_DIR = "/content/drive/MyDrive/ai_coding_models_uncensored"
FINETUNED_DIR = "/content/drive/MyDrive/ai_coding_models_finetuned"
os.makedirs(FINETUNED_DIR, exist_ok=True)
print(f"📁 Thư mục nguồn uncensored (Bước 1): {UNCENSORED_DIR}")
print(f"📁 Thư mục xuất model đã fine-tune (Bước 2): {FINETUNED_DIR}")

In [ ]:
# @title 3. Khởi tạo FastLanguageModel với Unsloth
from unsloth import FastLanguageModel
import torch, os

# @markdown Chọn model bạn muốn Fine-tune:
BASE_MODEL_CHOICE = "Qwen/Qwen2.5-Coder-7B-Instruct" # @param ["Qwen/Qwen2.5-Coder-7B-Instruct", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "deepseek-ai/DeepSeek-Coder-V2-Lite-Instruct"]
MAX_SEQ_LENGTH = 4096 # @param {type:"integer"}
LOAD_IN_4BIT = True # @param {type:"boolean"}

clean_name = BASE_MODEL_CHOICE.split("/")[-1]
UNCENSORED_DRIVE_PATH = os.path.join(UNCENSORED_DIR, f"{clean_name}-Heretic-Uncensored")

# Kiểm tra xem có model đã uncensor từ Bước 1 trên Drive không
if os.path.exists(UNCENSORED_DRIVE_PATH):
    LOAD_PATH = UNCENSORED_DRIVE_PATH
    print(f"📦 Tìm thấy Model đã Uncensor trên Google Drive: {LOAD_PATH}")
else:
    LOAD_PATH = BASE_MODEL_CHOICE
    print(f"🌐 Không thấy bản Uncensor trên Drive. Nạp trực tiếp từ Hugging Face: {LOAD_PATH}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = LOAD_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None, # Tự động chọn (Float16 cho T4/L4, Bfloat16 cho A100)
    load_in_4bit = LOAD_IN_4BIT,
)

# Cấu hình LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # LoRA Rank (16 hoặc 32 cho Code)
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, # Unsloth tối ưu 0 dropout để tăng tốc
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Cấu hình LoRA Adapter thành công!")

In [ ]:
# @title 4. Chuẩn bị Dataset Coding
from datasets import Dataset, load_dataset

# @markdown Số lượng mẫu dữ liệu để train (Mặc định 2000 mẫu để train nhanh):
SAMPLE_COUNT = 2000 # @param {type:"integer"}

# Template Chat tiêu chuẩn cho Coding
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = alpaca_prompt.format(instruction, input_text, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

# Nạp dataset mẫu từ Hugging Face
dataset = load_dataset("iamtarun/python_code_instructions_18k_alpaca", split = f"train[:{SAMPLE_COUNT}]")
dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"📊 Đã nạp thành công {len(dataset)} mẫu huấn luyện coding!")

In [ ]:
# @title 5. Tiến hành Huấn luyện (SFTTrainer)
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

MAX_STEPS = 60 # @param {type:"integer"} # Ghi chú: Đặt 60 để demo nhanh, hoặc 300-1000 cho model hội tụ tốt

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = MAX_STEPS,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Bắt đầu huấn luyện với Unsloth...")
trainer_stats = trainer.train()
print("🎉 Huấn luyện hoàn tất!")

In [ ]:
# @title 6. Kiểm tra Thử nghiệm Suy luận (Inference Test)
FastLanguageModel.for_inference(model)

test_prompt = alpaca_prompt.format(
    "Write a Python function to find all prime numbers up to n using Sieve of Eratosthenes.",
    "n = 50",
    ""
)
inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")

print("🤖 Đang sinh code thử nghiệm...")
outputs = model.generate(**inputs, max_new_tokens = 256, use_cache = True)
result = tokenizer.batch_decode(outputs)
print("="*60)
print(result[0])
print("="*60)

In [ ]:
# @title 7. Lưu Model đã Fine-tune vào Google Drive (Merged 16-bit)
OUTPUT_FINETUNED_PATH = os.path.join(FINETUNED_DIR, f"{clean_name}-Finetuned-Final")

print(f"💾 Đang lưu mô hình hợp nhất (Merged 16-bit) sang Google Drive: {OUTPUT_FINETUNED_PATH}...")
model.save_pretrained_merged(OUTPUT_FINETUNED_PATH, tokenizer, save_method = "merged_16bit")
print("✅ Lưu hoàn tất 100%! Model này đã sẵn sàng để nạp ở Bước 3 (Serving)!")